# Data Cleaning & Validation — Notebook Documentation

**Notebook:** `data_cleaning.ipynb`

**Purpose:** Evaluates the structural quality, completeness, and referential integrity of the synthetic Graduation Intelligence datasets before loading them into the PostgreSQL database. Identifies missing values, duplicate records, and invalid foreign key relationships to support reliable graduation readiness analysis.

**Record level:** Varies by source dataset. The notebook evaluates multiple datasets and generates dataset-level summary statistics.

**Input datasets:**

- `students.csv`: Student demographic and enrollment information.
- `course_catalogue.csv`: Available courses and associated credit values.
- `course_enrollments.csv`: Student course enrollments, grades, and credits earned.
- `graduation_requirements.csv`: Cohort-specific graduation credit requirements.
- `keystone_assessments.csv`: Student Keystone assessment administrations and scores.
- `act158_pathways.csv`: Proposed Act 158 graduation pathways and evidence statuses.
- `community_service.csv`: Student community service activities and verification statuses.
- `interventions.csv`: Student academic and graduation support interventions.

**Data source:** Synthetic datasets stored in the Google Drive directory `Graduation_Tracker/data/ground_truth`.

**Processing methodology:**

- Import pandas and NumPy for data processing and analysis.
- Connect Google Colab to Google Drive.
- Define the source directory using Python's `pathlib.Path`.
- Load all eight CSV datasets into pandas DataFrames.
- Create a dictionary containing seven datasets for initial data profiling.
- Calculate dataset-level summary statistics.
- Validate selected foreign key relationships using pandas membership checks.

**Data profiling fields:**

- `dataset`: Name of the dataset being evaluated.
- `rows`: Total number of records in the dataset.
- `columns`: Total number of fields in the dataset.
- `missing_values`: Total number of null values across all fields.
- `duplicates`: Number of completely duplicated rows.

**Business rules and validation checks:**

- All datasets must be readable as CSV files.
- Dataset dimensions are calculated using DataFrame row and column counts.
- Missing values are identified using `isnull()`.
- Duplicate records are identified using `duplicated()`.
- Course enrollment student identifiers must exist in the student dataset.
- Course enrollment course identifiers must exist in the course catalogue.
- Invalid student references are identified using `isin()` and Boolean negation.
- Invalid course references are identified using the same method.
- Foreign key validation results are reported as counts of invalid references.

**Data quality checks implemented:**

- Dataset row counts.
- Dataset column counts.
- Total missing values per dataset.
- Completely duplicated rows per dataset.
- Invalid student references in course enrollments.
- Invalid course references in course enrollments.

**Outputs:**

- `profile`: pandas DataFrame containing dataset-level profiling statistics.
- `invalid_students`: Boolean Series identifying course enrollment records with student IDs absent from the student dataset.
- `involid_courses`: Boolean Series identifying course enrollment records with course IDs absent from the course catalogue.
- Printed counts of invalid student and course references.

The notebook currently displays validation results but does not export a cleaned dataset or data quality report.

**Data cleaning status:** Initial data profiling and selected referential integrity checks are implemented. Missing values, duplicate records, and invalid references are identified but are not automatically corrected or removed.

**Limitations:** The notebook does not currently implement comprehensive data cleaning, missing-value treatment, duplicate removal, data type standardization, or validation of all foreign key relationships. The student dataset is loaded but excluded from the dataset profiling dictionary. Duplicate detection identifies completely identical rows rather than duplicate primary keys. Missing values are counted across entire datasets without distinguishing expected null values from data quality issues. Foreign key validation is limited to the student and course references in course enrollments. The notebook does not yet generate cleaned output files or a persistent data quality audit report.

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

In [ ]:
# Connect Google Drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create folder path

folder_path = Path('/content/drive/MyDrive/Graduation_Tracker/data/ground_truth')

In [ ]:
# Load datasets

pathways = pd.read_csv(folder_path/'act158_pathways.csv')
community_service = pd.read_csv(folder_path/'community_service.csv')
course_catalogue = pd.read_csv(folder_path/'course_catalogue.csv')
course_enrollment = pd.read_csv(folder_path/'course_enrollments.csv')
requirements = pd.read_csv(folder_path/'graduation_requirements.csv')
interventions = pd.read_csv(folder_path/'interventions.csv')
keystones = pd.read_csv(folder_path/'keystone_assessments.csv')
students = pd.read_csv(folder_path/'students.csv')

In [ ]:
# Review summaries of datasets

datasets = {
    'pathways': pathways,
    'community_service': community_service,
    'course_catalogue': course_catalogue,
    'course_enrollment': course_enrollment,
    'requirements': requirements,
    'interventions': interventions,
    'keystones': keystones,
}

summary = []

for name, df in datasets.items():
  summary.append({
      'dataset':name,
      'rows': len(df),
      'columns': len(df.columns),
      'missing_values': df.isnull().sum().sum(),
      'duplicates': df.duplicated().sum()
  })

  profile = pd.DataFrame(summary)

  display(profile)

,dataset,rows,columns,missing_values,duplicates
0,pathways,5000,5,1949,0


,dataset,rows,columns,missing_values,duplicates
0,pathways,5000,5,1949,0
1,community_service,9913,5,0,0


,dataset,rows,columns,missing_values,duplicates
0,pathways,5000,5,1949,0
1,community_service,9913,5,0,0
2,course_catalogue,67,4,0,0


,dataset,rows,columns,missing_values,duplicates
0,pathways,5000,5,1949,0
1,community_service,9913,5,0,0
2,course_catalogue,67,4,0,0
3,course_enrollment,85618,10,34802,0


,dataset,rows,columns,missing_values,duplicates
0,pathways,5000,5,1949,0
1,community_service,9913,5,0,0
2,course_catalogue,67,4,0,0
3,course_enrollment,85618,10,34802,0
4,requirements,36,3,0,0


,dataset,rows,columns,missing_values,duplicates
0,pathways,5000,5,1949,0
1,community_service,9913,5,0,0
2,course_catalogue,67,4,0,0
3,course_enrollment,85618,10,34802,0
4,requirements,36,3,0,0
5,interventions,9879,5,0,0


,dataset,rows,columns,missing_values,duplicates
0,pathways,5000,5,1949,0
1,community_service,9913,5,0,0
2,course_catalogue,67,4,0,0
3,course_enrollment,85618,10,34802,0
4,requirements,36,3,0,0
5,interventions,9879,5,0,0
6,keystones,12975,7,0,0


In [ ]:
# Foreign key validation

invalid_students = ~course_enrollment['student_id'].isin(students['student_id'])

involid_courses = ~course_enrollment['course_id'].isin(course_catalogue['course_id'])

print(invalid_students.sum())
print(involid_courses.sum())

0
0
